part 1: preparing the data, one-hot encoding

In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank = fetch_ucirepo(id=222)
X = bank.data.features.drop(columns=["duration"])
y = (bank.data.targets["y"] == "yes").astype(int)

X = pd.get_dummies(X)
print(X.shape)
print(X.columns.tolist())

(45211, 46)
['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'marital_divorced', 'marital_married', 'marital_single', 'education_primary', 'education_secondary', 'education_tertiary', 'default_no', 'default_yes', 'housing_no', 'housing_yes', 'loan_no', 'loan_yes', 'contact_cellular', 'contact_telephone', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_failure', 'poutcome_other', 'poutcome_success']


part 2: Splitting into train and test

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(36168, 46) (9043, 46)
0.11698186241981863 0.11699657193409267


part 3: Training the model

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

jobs = bank.data.features.loc[X_test.index, "job"]

for i in range(10):
    age = X_test["age"].iloc[i]
    job = jobs.iloc[i]
    balance = X_test["balance"].iloc[i]
    chance = probs[i] * 100
    real = "yes" if y_test.iloc[i] == 1 else "no"
    print(f"Customer {i+1}: {age}-year-old {job}, balance €{balance:,}. Model says {chance:.0f}% chance of yes. Really said: {real}.")

Customer 1: 40-year-old blue-collar, balance €640. Model says 1% chance of yes. Really said: no.
Customer 2: 44-year-old technician, balance €378. Model says 3% chance of yes. Really said: no.
Customer 3: 31-year-old services, balance €356. Model says 3% chance of yes. Really said: no.
Customer 4: 36-year-old blue-collar, balance €655. Model says 2% chance of yes. Really said: no.
Customer 5: 34-year-old services, balance €1,921. Model says 12% chance of yes. Really said: no.
Customer 6: 41-year-old management, balance €1,138. Model says 5% chance of yes. Really said: no.
Customer 7: 43-year-old entrepreneur, balance €2,589. Model says 9% chance of yes. Really said: no.
Customer 8: 64-year-old retired, balance €109. Model says 80% chance of yes. Really said: yes.
Customer 9: 35-year-old admin., balance €1,803. Model says 16% chance of yes. Really said: no.
Customer 10: 44-year-old services, balance €2,279. Model says 8% chance of yes. Really said: no.


c:\Users\USER\OneDrive\Desktop\bank-recommendation-engine\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
